In [0]:
    
def validate_star_schema():
    fact = spark.table("workspace.dev_gold_layer.fact_sales")
    dim = spark.table("workspace.dev_gold_layer.dim_products")
    # Test: Every product_id in Fact should exist in Dimension (Referential Integrity)
    orphans = fact.join(dim, "product_id", "left_anti").count()
    assert orphans == 0, f"ERROR: Found {orphans} sales records with no product description!"
    
    print("Star Schema Integrity Verified.")

validate_star_schema()

In [0]:
from pyspark.sql.functions import sum, col

def validate_gold():
    # 1. Read the tables
    silver_df = spark.table("workspace.dev_silver_layer.events_cleaned")
    gold_df = spark.table("workspace.dev_gold_layer.fact_sales")
    
    # 2. Calculate Expected Revenue (Silver)
    # Use float() and coalesce with 0.0 to prevent 'None' or 'Str' errors
    res_silver = silver_df.filter("event_type = 'purchase' AND brand IS NOT NULL") \
                          .select(sum(col("price").cast("double"))).collect()[0][0]
    expected_rev = float(res_silver) if res_silver is not None else 0.0
        
    # 3. Calculate Actual Revenue (Gold)
    res_gold = gold_df.select(sum(col("total_revenue").cast("double"))).collect()[0][0]
    actual_rev = float(res_gold) if res_gold is not None else 0.0
    
    print(f"DEBUG: Expected Revenue: {expected_rev} | Actual Gold Revenue: {actual_rev}")
    
    # 4. Perform Assertion
    # Check if the difference is less than 1 dollar
    assert abs(expected_rev - actual_rev) < 1.0, f"Mismatch! Diff: {expected_rev - actual_rev}"
    
    print(f"Gold Validation Passed. Total Revenue verified: ${actual_rev:,.2f}")

validate_gold()